# **Modeling**

**Mục tiêu:** Xây dựng mô hình Softmax Regression nhằm phân loại người dùng rời bỏ và ở lại dịch vụ tín dụng dựa trên các đặc trưng của tập dataset đã tiền xử lý trước đó

## *Import thư viện cần thiết*

In [1]:
import sys
import os
import numpy as np
from IPython.display import display, Markdown
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

%load_ext autoreload
%autoreload 2

from src.models import SoftmaxRegression

## *1. Xây dựng tập train và test từ dataset đã tiền xử lý*

### 1.1 Tải dữ liệu

In [2]:
data = np.genfromtxt('../data/processed/bank_churners_preprocessed.csv', delimiter=',', skip_header=1)

data.shape

(10127, 38)

### 1.2 Chia thành tập đặc trưng và tập nhãn

In [3]:
X = data[:, :-1]

y = data[:, -1].astype(int)

X.shape, y.shape

((10127, 37), (10127,))

### 1.3 Chia thành 2 tập train và test với tỉ lệ 8 train / 2 test

In [4]:
n_samples = X.shape[0]

# Tạo danh sách index và xáo trộn ngẫu nhiên
indices = np.arange(n_samples)
np.random.seed(42) # Cố định seed để kết quả giống nhau mỗi lần chạy
np.random.shuffle(indices)

# Tính vị trí cắt
split_index = int(n_samples * 0.8)

# Lấy index cho từng tập train/test
train_indices = indices[:split_index]
test_indices = indices[split_index:]

X_train = X[train_indices]
y_train = y[train_indices]
X_test = X[test_indices]
y_test = y[test_indices]

print(f'Train: {X_train.shape, y_train.shape}')
print(f'Test: {X_test.shape, y_test.shape}')

Train: ((8101, 37), (8101,))
Test: ((2026, 37), (2026,))


## *2. Huấn luyện mô hình*

### 2.1 Khởi tạo mô hình

In [5]:
model = SoftmaxRegression(in_features=X_train.shape[1], out_features=2)

### 2.2 Cài đặt các tham số trước khi huấn luyện

In [6]:
EPOCHS = 10000
LEARNING_RATE = 0.01

### 2.3 Huấn luyện mô hình

In [7]:
model.fit(X=X_train, y=y_train, epochs=EPOCHS, lr=LEARNING_RATE)

Epoch 0, Loss: 0.6827


Epoch 100, Loss: 0.3909
Epoch 200, Loss: 0.3365
Epoch 300, Loss: 0.3123
Epoch 400, Loss: 0.2983
Epoch 500, Loss: 0.2890
Epoch 600, Loss: 0.2824
Epoch 700, Loss: 0.2773
Epoch 800, Loss: 0.2734
Epoch 900, Loss: 0.2702
Epoch 1000, Loss: 0.2675
Epoch 1100, Loss: 0.2652
Epoch 1200, Loss: 0.2632
Epoch 1300, Loss: 0.2615
Epoch 1400, Loss: 0.2599
Epoch 1500, Loss: 0.2585
Epoch 1600, Loss: 0.2573
Epoch 1700, Loss: 0.2561
Epoch 1800, Loss: 0.2550
Epoch 1900, Loss: 0.2540
Epoch 2000, Loss: 0.2531
Epoch 2100, Loss: 0.2522
Epoch 2200, Loss: 0.2514
Epoch 2300, Loss: 0.2506
Epoch 2400, Loss: 0.2499
Epoch 2500, Loss: 0.2492
Epoch 2600, Loss: 0.2485
Epoch 2700, Loss: 0.2479
Epoch 2800, Loss: 0.2473
Epoch 2900, Loss: 0.2467
Epoch 3000, Loss: 0.2462
Epoch 3100, Loss: 0.2457
Epoch 3200, Loss: 0.2452
Epoch 3300, Loss: 0.2447
Epoch 3400, Loss: 0.2442
Epoch 3500, Loss: 0.2438
Epoch 3600, Loss: 0.2434
Epoch 3700, Loss: 0.2430
Epoch 3800, Loss: 0.2426
Epoch 3900, Loss: 0.2422
Epoch 4000, Loss: 0.2419
Epoch 410

### 2.4 Đánh giá mô hình

In [8]:
def calculate_metrics(y_true, y_pred):
    # Number of classes
    classes = np.unique(np.concatenate((y_true, y_pred)))
    n_classes = len(classes)
    
    # Create Confusion Matrix
    conf_matrix = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        conf_matrix[int(t)][int(p)] += 1
        
    # Accuracy
    accuracy = np.trace(conf_matrix) / np.sum(conf_matrix)
    
    results = {
        "overall_metrics": {
            "accuracy": float(accuracy)
        },
        "per_class_metrics": {},
        "confusion_matrix": conf_matrix.tolist() 
    }
    
    precisions = []
    recalls = []
    f1_scores = []
        
    # Calculate Precision, Recall, F1 for each classification class
    for i in range(n_classes):
        tp = conf_matrix[i, i]
        fp = np.sum(conf_matrix[:, i]) - tp
        fn = np.sum(conf_matrix[i, :]) - tp
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0
        
        precisions.append(p)
        recalls.append(r)
        f1_scores.append(f1)
        
        results["per_class_metrics"][f"class_{i}"] = {
            "precision": float(p),
            "recall": float(r),
            "f1_score": float(f1)
        }

    # Calculate macro-averaged metrics
    results["overall_metrics"]["macro_precision"] = float(np.mean(precisions))
    results["overall_metrics"]["macro_recall"] = float(np.mean(recalls))
    results["overall_metrics"]["macro_f1"] = float(np.mean(f1_scores))

    return results

In [9]:
results = calculate_metrics(y_test, y_pred=model.predict(X_test))

md_output = "## Báo Cáo Kết Quả Mô Hình (Model Performance)\n"
md_output += "---\n"

acc = results['overall_metrics']['accuracy']
md_output += f"### 1. Tổng quan\n"
md_output += f"- **Accuracy (Độ chính xác toàn cục):** `{acc:.4f}` ({acc*100:.2f}%)\n\n"

md_output += "### 2. Confusion Matrix\n"
matrix = results['confusion_matrix']
n_classes = len(matrix)

header = "| Actual \\ Predicted | " + " | ".join([f"Pred {i}" for i in range(n_classes)]) + " |\n"
separator = "|---|" + "---|" * n_classes + "\n"
md_output += header + separator

for i, row in enumerate(matrix):
    row_str = " | ".join(map(str, row))
    md_output += f"| **Actual {i}** | {row_str} |\n"

md_output += "\n"

if results['per_class_metrics']:
    md_output += "### 3. Chỉ số chi tiết từng lớp (Per-class Metrics)\n"
    
    first_class_key = list(results['per_class_metrics'].keys())[0]
    metric_names = list(results['per_class_metrics'][first_class_key].keys())
    
    p_header = "| Class | " + " | ".join([m.capitalize() for m in metric_names]) + " |\n"
    p_sep = "|---|" + "---|" * len(metric_names) + "\n"
    md_output += p_header + p_sep
    
    for cls_name, metrics in results['per_class_metrics'].items():
        vals = [f"{metrics[m]:.4f}" for m in metric_names]
        md_output += f"| **{cls_name}** | {' | '.join(vals)} |\n"

display(Markdown(md_output))

## Báo Cáo Kết Quả Mô Hình (Model Performance)
---
### 1. Tổng quan
- **Accuracy (Độ chính xác toàn cục):** `0.9102` (91.02%)

### 2. Confusion Matrix
| Actual \ Predicted | Pred 0 | Pred 1 |
|---|---|---|
| **Actual 0** | 1648 | 55 |
| **Actual 1** | 127 | 196 |

### 3. Chỉ số chi tiết từng lớp (Per-class Metrics)
| Class | Precision | Recall | F1_score |
|---|---|---|---|
| **class_0** | 0.9285 | 0.9677 | 0.9477 |
| **class_1** | 0.7809 | 0.6068 | 0.6829 |
